# Update T-Route channel geometry: Arkansas-White-Red region (HUC2 11), a = 0.4

T-Route's Muskingum-Cunge routing treats every reach as a trapezoidal channel, defined in the NWM RouteLink file. This notebook replaces that geometry with ML-predicted bankfull geometry from [HydroGeom](https://github.com/Reizrb/HydroGeom) for the reaches in **HUC2 11**, using a trapezoid whose bottom width is **0.4 × top width**, and recomputes the Muskingum parameters.

The original NWM RouteLink file is downloaded from NOAA automatically and is never modified; a new file is written.

### STEP 1. Install and import

Install once:
```bash
pip install numpy pandas netCDF4 "git+https://github.com/Reizrb/HydroGeom"
```

In [1]:
import sys
sys.path.append("..")          # troute_geometry.py is in the repository's main folder

import pandas as pd
import troute_geometry as tg

### STEP 2. Settings

- `a`: bottom width / top width of the trapezoid (0 = triangle; use 0.99 for a nearly rectangular channel)
- `huc2`: region whose reaches get the new geometry (all other reaches keep the original geometry)

In [2]:
a = 0.4
huc2 = "11"

routelink_in = "RouteLink_CONUS.nc"
routelink_out = f"RouteLink_CONUS_HUC2-{huc2}_a{a}.nc"

### STEP 3. Original NWM RouteLink file

Downloaded from NOAA's server for the current operational NWM release (skipped if the file is already here). To use a specific file instead, set `routelink_in` to its path in STEP 2.

In [3]:
tg.download_routelink(routelink_in)

Downloaded RouteLink_CONUS.nc (269 MB) from https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.1.6/parm/domain/RouteLink_CONUS.nc


'RouteLink_CONUS.nc'

### STEP 4. Bankfull geometry from HydroGeom

Bankfull width and depth for every NHDPlusV2.1 reach in the region. For all of CONUS, download the dataset from [Zenodo](https://doi.org/10.5281/zenodo.19208847) and use `tg.read_geometry_file(path)` instead.

In [4]:
geometry = tg.channel_geometry(huc2=huc2)
print(f"{len(geometry):,} reaches with bankfull geometry in HUC2 {huc2}")

204,979 reaches with bankfull geometry in HUC2 11


### STEP 5. Write the updated RouteLink file

Sets `BtmWdth`, `TopWdth`, and `ChSlp` for the trapezoid, and recomputes `TopWdthCC`, `MusK`, and `MusX`.

In [5]:
tg.update_routelink(routelink_in, routelink_out, geometry, a=a)

Updated 204,979 reaches (a = 0.4); 2,571,755 RouteLink reaches kept their geometry; 0 geometry reaches are not in the RouteLink file.


{'updated': 204979, 'not_in_routelink': 0, 'unchanged': 2571755}

### STEP 6. Original vs updated values for the reaches in the region

In [6]:
pd.set_option("display.width", 160)
tg.compare(routelink_in, routelink_out, comids=geometry["COMID"])

median               mean                 min               max            
file       original  updated  original   updated  original updated  original     updated
variable                                                                                
BtmWdth       2.783    8.020     5.563     9.841     0.135   2.925   117.296     289.790
TopWdth       4.639   20.051     9.272    24.602     0.225   7.313   195.493     724.476
ChSlp         0.593    0.219     0.582     0.220     0.114   0.029     2.254       0.674
TopWdthCC    13.922   60.152    28.669    73.806     0.674  21.940   586.480    2173.427
MusK       3600.000  899.192  3600.000  1906.023  3600.000   0.042  3600.000  162835.422
MusX          0.200    0.465     0.200     0.350     0.200   0.000     0.200       0.500

### STEP 7. Use it in T-Route

Point T-Route to the new file in its configuration YAML:

```yaml
network_topology_parameters:
  supernetwork_parameters:
    geo_file_path: RouteLink_CONUS_HUC2-11_a0.4.nc
```